# 🎯 Single & Multi-Turn Dataset Execution with YAML Config

This notebook uses **PyRIT** to run multi‐turn simulations (or red‐teaming) against an HTTP‐based assistant.  
All parameters (dataset path, evaluator path, raw HTTP template, response‐parsing fields, thread‐ID regex) live in a single `dataset_config.yaml`.  

**Workflow**:
1. Load environment variables and `dataset_config.yaml`
2. Build a multi‐field response parser
3. Instantiate `HTTPTargetX` with our raw HTTP template + parser
4. Instantiate an `Evaluator` (scorer) and `PromptSendingAttack`
5. Run the orchestrator over every QA pair in `dataset.yaml`
6. Generate a HTML report at the end

> **Tip for teams**:
> - As your endpoint, models, or parsing rules evolve, change only `dataset_config.yaml`.  
> - The notebook code remains unchanged.


In [1]:
# Cell 1: Load environment variables, imports, and config

from dotenv import load_dotenv
import os
import re
import time
import asyncio
import yaml
from pathlib import Path
from datetime import datetime

import logging
logging.basicConfig(level=logging.WARNING)

from pyrit.common import IN_MEMORY, initialize_pyrit
from pyrit.prompt_target import MultiFieldResponseParser, OpenAIChatTarget
from pyrit.prompt_target import HTTPTargetX
from pyrit.score import Evaluator
from typing import Optional

# Initialize PyRIT memory (in‐memory DuckDB)
initialize_pyrit(memory_db_type=IN_MEMORY)

# Load our configuration from config.yaml
config_path = Path("assets/demo_scorer_definitions/dataset_config.yaml")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

# Extract paths + raw template + parser definitions
dataset_path       = cfg["dataset_path"]       # e.g. "dataset.yaml"
evaluator_path     = cfg["evaluator_path"]     # e.g. "scorer.yaml"
raw_template       = cfg["http_request_raw"]   # multi‐line string with {{PROMPT}}
field_defs         = cfg["field_defs"]         # list of dicts: name/type/pattern
thread_id_pattern  = cfg["thread_id_pattern"]
thread_id_key      = cfg["thread_id_query_param_key"]

# Load .env (for credentials, endpoint base URL, etc.)
load_dotenv()
base_url   = os.getenv("TARGET_ENDPOINT")
token      = os.getenv("AUTH_TOKEN")

# Substitute {base_url} and {token} into the raw HTTP template.
# Because the YAML uses double‐braces around {{PROMPT}}, Python .format() will produce a single {PROMPT} placeholder.
http_request_templated = raw_template.format(
    base_url=base_url,
    token=token
)

## 🔍 Build the Response Parser and Thread‐ID Helpers

1. **`MultiFieldResponseParser`**:  
   Reads `field_defs` from YAML, so you can add or remove fields without changing code.

2. **`thread_id_parser`**:  
   Uses the regex in `thread_id_pattern` from YAML to extract any newly created thread IDs.

3. **`thread_id_injector`**:  
   A small helper that appends thread Id to the original URL.


In [2]:
# Cell 2: Build MultiFieldResponseParser and thread‐ID helpers

import requests

# 2.1 Instantiate MultiFieldResponseParser from YAML‐loaded field_defs
multi_parser = MultiFieldResponseParser(field_definitions=field_defs)

# 2.2 thread_id_parser using the YAML‐provided regex
def thread_id_parser(response: str) -> Optional[str]:
    match = re.search(thread_id_pattern, response)
    return match.group(1) if match else None

# 2.3 thread_id_injector (appends threadId received to the next request)
def thread_id_injector(raw_http_request: str, thread_id: str, thread_id_key: str = "threadId") -> str:
    """
    Injects or replaces the `{thread_id_key}` query parameter in the first URL found in the raw HTTP request.
    """
    import re

    url_pattern = r"(https?://[^\s]+)"
    m = re.search(url_pattern, raw_http_request)
    if not m:
        raise ValueError("No URL found in raw HTTP request; cannot inject thread ID.")

    original_url = m.group(1)

    # Remove existing threadId (or other key) if present
    pattern = rf"([?&]){re.escape(thread_id_key)}=[^&]*"
    cleaned = re.sub(pattern, "", original_url)

    sep = "&" if "?" in cleaned else "?"
    new_url = f"{cleaned}{sep}{thread_id_key}={thread_id}"

    return raw_http_request.replace(original_url, new_url)




## 🚀 Instantiate HTTPTargetX, Evaluator, and PromptSendingAttack

1. **`HTTPTargetX`**:  
   • `http_request` = `http_request_templated` (from Cell 2).  
   • `prompt_regex_string` = `{PROMPT}` (standard placeholder).  
   • `response_parser` = `multi_parser` (streams → JSON/regex).  
   • `thread_id_parser` = our helper (Cell 4).

2. **`Evaluator`**:  
   Uses `OpenAIChatTarget` plus your evaluator YAML (`evaluator_path`).  
   Set `scorer_type = "float_scale"` or `"true_false"` as needed.

3. **`PromptSendingOrchestrator`**:
   • `objective_target` = `HTTPTargetX` instance  
   • `objective_scorer` = `scorer` (Evaluator instance)  
   • `thread_id_injector` = our helper (Cell 4)  
   • `batch_size = 1`, `retries_on_objective_failure = 0`, `verbose = True`  


In [3]:
# Cell 3: Instantiate HTTPTargetX, Evaluator, and PromptSendingOrchestrator
import aiohttp

# Set robust timeout values (in seconds)
timeout = aiohttp.ClientTimeout(
    total=None,           # No overall timeout, or set to e.g. 300 for 5 min
    connect=60,           # 60 seconds to connect
    sock_connect=60,      # 60 seconds to establish a socket connection
    sock_read=300         # 300 seconds to read response (for streaming endpoints)
)

client = aiohttp.ClientSession(timeout=timeout)

# 3.1 Create our HTTP target using the templated raw request
http_prompt_target = HTTPTargetX(
    http_request         = http_request_templated,
    prompt_regex_string  = "{PROMPT}",
    use_tls              = True,
    response_parser      = multi_parser,
    thread_id_parser     = thread_id_parser,
    client               = client
)

# 3.2 Build an Evaluator (scorer) from the YAML path
scorer = Evaluator(
    chat_target           = OpenAIChatTarget(),
    evaluator_yaml_path   = Path(evaluator_path),
    scorer_type           = "float_scale"
)

# 3.3 Instantiate the PromptSendingOrchestrator
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.attacks import SingleTurnAttackContext, PromptSendingAttack, AttackConverterConfig, AttackScoringConfig

attack = PromptSendingAttack(
    objective_target=http_prompt_target,
    attack_converter_config=AttackConverterConfig(
        request_converters=[], 
        response_converters=[]
    ),
    attack_scoring_config=AttackScoringConfig(
        objective_scorer=scorer,
        auxiliary_scorers=[]
    ),
    max_attempts_on_failure=0,
    prompt_normalizer=PromptNormalizer()  # Or set as needed, optional
)

## 📝 Define Report Generation Helper

We will run the orchestrator on each QA example, then produce a **dataset report** (HTML) at the end.  
This function:

- Ensures the report directory exists.
- Names the report `dataset_report_<timestamp>.html`.
- Calls `generate_dataset_report(...)` from `pyrit.common.text_helper`.

Every conversation’s transcript, assistant responses, and scores will be included in the HTML.

In [4]:
# Cell 4: Define report‐generation helper

async def generate_report(results: list, execution_time: float):
    """
    Saves a dataset-style HTML report for all conversation results.
    """
    # Report directory—same parent folder as the dataset file
    report_dir = Path(dataset_path).parent.resolve()
    report_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"dataset_report_{timestamp}.html"

    generate_dataset_report(
        results        = results,
        save_path      = report_dir / filename,
        description    = (
            "Mixed evaluation of single-turn and multi-turn prompt requests. "
            "Lowest step score is used to indicate the final scenario score."
        ),
        execution_time = execution_time
    )

## 🏃 Main Async Runner

1. Load all QA pairs from `dataset_path` (via `pyrit.loaders.loader.load_test_data`).
2. Call `await orchestrator.execute(qa_pairs)` to run each example.
3. After completion, retrieve all chat results via `orchestrator.get_all_chat_results()`.
4. Generate the HTML report.

In [ ]:
# Cell 5: Main async runner

from loaders.loader import load_test_data
from pyrit.common.report_generator import get_conversation_report_async
from pyrit.common.report_generator import create_report

async def main():
    start_time = time.time()

    # 5.1 Load all QA pairs from the dataset YAML
    qa_pairs = load_test_data(f"{dataset_path}")

    # 5.2 Execute attack over every example in qa_pairs
    attack_results = await attack.perform_dataset_attack(qa_pairs, thread_id_injector=thread_id_injector)

    # 5.3 Gather results and measure elapsed time
    chat_reports = []
    for attack_result in attack_results:
        report = await get_conversation_report_async(attack_result)
        chat_reports.append(report)
        
    execution_time = time.time() - start_time

    # 5.4 Generate the HTML report
    html = create_report(results= chat_reports, execution_time= execution_time, description= "This report provides an overview of the dataset test cases executed in real-time through the target application.")
    
# Kick off
await main()

HTTP Response: event:THREAD_CREATED
data:thread_oOHD70QfreLkZhB8et2NqUgg

event:SUMMARY_MESSAGE
data:Kip en pasta recepten

event:TEXT_MESSAGE
data:Het

event:TEXT_MESSAGE
data: klinkt

event:TEXT_MESSAGE
data: alsof

event:TEXT_MESSAGE
data: je

event:TEXT_MESSAGE
data: zin

event:TEXT_MESSAGE
data: hebt

event:TEXT_MESSAGE
data: in

event:TEXT_MESSAGE
data: een

event:TEXT_MESSAGE
data: heerlijke

event:TEXT_MESSAGE
data: maaltijd

event:TEXT_MESSAGE
data: met

event:TEXT_MESSAGE
data: kip

event:TEXT_MESSAGE
data: en

event:TEXT_MESSAGE
data: pasta

event:TEXT_MESSAGE
data:!

event:TEXT_MESSAGE
data: L

event:TEXT_MESSAGE
data:aten

event:TEXT_MESSAGE
data: we

event:TEXT_MESSAGE
data: wat

event:TEXT_MESSAGE
data: recepten

event:TEXT_MESSAGE
data: voor

event:TEXT_MESSAGE
data: je

event:TEXT_MESSAGE
data: vinden

event:TEXT_MESSAGE
data:.

event:TEXT_MESSAGE
data:
data:

event:TEXT_MESSAGE
data:Ik heb heerlijke kip en pasta recepten voor je gevonden. Geniet van de romige en smaak